# 01 Collect Yearly Data

Canonical data collection notebook for the refactored AMR workflow.
This notebook builds yearly pair counts, yearly background marginals, runs the unified LLM screen, and exports `yearly_amr_screened_pairs.parquet`.


In [ ]:
from pathlib import Path
import pandas as pd
from amr.filtering import (
    UNIFIED_AMR_INTEREST_SYSTEM_PROMPT,
    build_user_prompt,
    build_yearly_screened_pairs,
    prepare_yearly_candidate_pmids,
    run_unified_llm_filter,
)
from amr.llm import OpenAIConfig, OpenAILanguageModel
from amr.pipelines.yearly_collection import (
    collect_yearly_amr_dataset,
    collect_yearly_background_counts,
)
from amr.pubmed.queries import (
    build_antibiotic_pathogen_query,
    build_pathogen_tiab_query,
    prepare_antibiotic_pubmed_query_tiab,
)


## Glossary

- **AMR**: Antimicrobial Resistance.
- **SI**: Scientific Interest score, defined as residual AMR attention after background adjustment.
- **TIAB**: Title/Abstract PubMed field selector (search terms constrained to title or abstract text).
- **PMID**: PubMed Identifier for a publication record.
- **LLM**: Large Language Model used for abstract screening.
- **BG / background counts**: Yearly publication marginals for pathogens or antibiotics used in SI baseline modeling.
- **Slice**: A chosen year interval (`SLICE_START_YEAR` to `SLICE_END_YEAR`) processed in one run.


## Configuration
Set paths and run options. Use a real PubMed contact email and valid LLM credentials.


In [ ]:
PATHOGENS_PATH = Path("bacteria_synonym_lookup.csv")
ANTIBIOTIC_METADATA_PATH = Path("antibiotic_list2.xlsx")
OUTPUT_YEARLY_AMR_PATH = Path("yearly_amr_pairs.parquet")
OUTPUT_YEARLY_SCREEN_PATH = Path("yearly_amr_screened_pairs.parquet")
OUTPUT_UNIFIED_DECISIONS_PATH = Path("yearly_amr_unified_decisions.parquet")
OUTPUT_PATHOGEN_BG_PATH = Path("pathogen_tiab_counts.parquet")
OUTPUT_ANTIBIOTIC_BG_PATH = Path("antibiotic_tiab_counts.parquet")
INTERVAL_PMIDS_PATH = Path("pmids_results.parquet")
YEARLY_PAIR_PMID_PATH = Path("yearly_pair_pmid.parquet")
PUBMED_METADATA_CACHE_PATH = Path("pubmed_pmid_metadata.parquet")
ABSTRACT_CACHE_PATH = Path("pmid_abstracts.parquet")
PUBMED_EMAIL = "xfcosta@gmail.com"  # replace with a real contact email
# Partition-friendly execution window: run a slice now, merge into canonical stores.
SLICE_START_YEAR = 2020
SLICE_END_YEAR = 2026
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"
INTERVAL_YEAR_GAP = 5
# Existing notebook intent: AMR-focused retrieval before unified LLM relevance screen.
EXTRA_AMR_KEYWORDS = [
    '"antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract]'
]
RUN_LLM_FILTER = True
MAX_LLM_CALLS = None
# Editable per-run prompt (single source of truth for this notebook execution)
UNIFIED_SYSTEM_PROMPT = UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
# Throughput/reliability controls for unified LLM filter
LLM_MAX_WORKERS = 2
LLM_BATCH_SIZE = 8
FAILED_RETRY_PATH = Path("yearly_amr_unified_failures.parquet")
MAX_FAILURE_RETRIES = 5
ABSTRACT_FETCH_BATCH_SIZE = 200
PREFETCH_ABSTRACTS = True
# Batch-style collection logging controls
COLLECTION_VERBOSE = 1  # 0=silent, 1=batch summaries, 2=detailed per interval
COLLECTION_PROGRESS_EVERY = 300
# Unified filter logging cadence
UNIFIED_PROGRESS_EVERY = 200


In [ ]:
from textwrap import fill

print("PRETTY PRINT (80 COLS) OF UNIFIED_SYSTEM_PROMPT:\n")
print("\n".join(fill(line, width=80) if line.strip() else "" for line in UNIFIED_SYSTEM_PROMPT.splitlines()))


## Load Pathogen and Antibiotic Lists


In [ ]:
bacteria_synonym_df = pd.read_csv(PATHOGENS_PATH)
antibiotic_df = pd.read_excel(ANTIBIOTIC_METADATA_PATH)

pathogens = sorted(bacteria_synonym_df["bacteria name"].dropna().astype(str).str.strip().unique())
antibiotics = sorted(antibiotic_df["Antibiotic"].dropna().astype(str).str.strip().unique())

print(f"Pathogens: {len(pathogens):,}")
print(f"Antibiotics: {len(antibiotics):,}")


## Collect Yearly Pair Dataset and Background Marginals


## PubMed Query Logic for Abstract Collection

The collection query is assembled per `(pathogen, antibiotic)` pair with four components:

1. **Antibiotic clause**: antibiotic name in `Abstract`, including split variants for hyphen/slash names (for example `co-trimoxazole` as exact phrase OR split terms).
2. **Pathogen clause**: pathogen + known synonyms in `Abstract` (`OR`-joined).
3. **AMR clause**: additional AMR-focused condition from `EXTRA_AMR_KEYWORDS` in `Title/Abstract`.
4. **Date clause**: publication date restricted to `[MIN_DATE, MAX_DATE]`.

Final query shape:

`(antibiotic_clause AND pathogen_clause) AND extra_amr_clause AND date_range_clause`


In [ ]:
sample_pathogen = pathogens[0]
sample_antibiotic = antibiotics[0]
sample_query = build_antibiotic_pathogen_query(
    sample_antibiotic,
    sample_pathogen,
    bacteria_synonym_df,
    extra_clauses=EXTRA_AMR_KEYWORDS,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
)
print("Sample pathogen:", sample_pathogen)
print("Sample antibiotic:", sample_antibiotic)
print("\nSample PubMed query:\n")
print(sample_query)


In [ ]:
print("EXAMPLE PUBMED QUERIES (MULTIPLE VIEWS)\n")

example_pairs = list(zip(pathogens[:3], antibiotics[:3]))
for idx, (p_name, a_name) in enumerate(example_pairs, start=1):
    pathogen_clause = build_pathogen_tiab_query(p_name, bacteria_synonym_df)
    antibiotic_clause = prepare_antibiotic_pubmed_query_tiab(a_name)
    pair_query = build_antibiotic_pathogen_query(
        a_name,
        p_name,
        bacteria_synonym_df,
        extra_clauses=EXTRA_AMR_KEYWORDS,
        min_date=MIN_DATE,
        max_date=MAX_DATE,
    )
    print(f"Example {idx}: {p_name} vs {a_name}")
    print("- Pathogen TIAB clause:")
    print(pathogen_clause)
    print("- Antibiotic TIAB clause:")
    print(antibiotic_clause)
    print("- Final pair + AMR + date query:")
    print(pair_query)
    print("\n" + "-" * 100 + "\n")


In the next code cell, the notebook will automatically gather research paper IDs from PubMed for each pathogen-antibiotic combination in your selected year range, convert those papers into yearly counts, and then save the results to local files so future runs can continue from where you left off. It also collects yearly background publication totals for each pathogen and antibiotic, which are later used to calculate scientific interest in a fair, comparable way.


In [ ]:
yearly_amr_df, yearly_pair_pmid_df, enriched_pair_pmid_df = collect_yearly_amr_dataset(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    email=PUBMED_EMAIL,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_output_path=YEARLY_PAIR_PMID_PATH,
    metadata_cache_path=PUBMED_METADATA_CACHE_PATH,
    output_path=OUTPUT_YEARLY_AMR_PATH,
    extra_keywords=EXTRA_AMR_KEYWORDS,
    collection_verbose=COLLECTION_VERBOSE,
    collection_progress_every=COLLECTION_PROGRESS_EVERY,
)
pathogen_bg_df, antibiotic_bg_df = collect_yearly_background_counts(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    start_year=SLICE_START_YEAR,
    end_year=SLICE_END_YEAR,
    email=PUBMED_EMAIL,
    pathogen_output_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_output_path=OUTPUT_ANTIBIOTIC_BG_PATH,
)
print(f"yearly AMR rows: {len(yearly_amr_df):,}")
print(f"pathogen bg rows: {len(pathogen_bg_df):,}")
print(f"antibiotic bg rows: {len(antibiotic_bg_df):,}")


## Unified LLM Filtering
Single strict pass: AMR evidence + scientific-interest relevance.


## Unified LLM Prompt Templates
This workflow uses a single strict LLM screen:
- **System prompt**: policy for AMR evidence + scientific-interest relevance.
- **User prompt template**: PMID-specific payload (pathogen, antibiotic, title, abstract).
**Prompt governance note:** if you change this prompt between slices, document it so downstream results remain interpretable.


In [ ]:
print("UNIFIED SYSTEM PROMPT:\n")
print(UNIFIED_SYSTEM_PROMPT)
print("\n" + "=" * 80 + "\n")
print("USER PROMPT TEMPLATE EXAMPLE:\n")
print(
    build_user_prompt(
        pmid="12345678",
        pathogen="Escherichia coli",
        antibiotic="Ciprofloxacin",
        title="Template title",
        abstract="Template abstract text.",
    )
)


In [ ]:
candidate_df = prepare_yearly_candidate_pmids(yearly_amr_df)
candidate_df = candidate_df.loc[
    candidate_df["year"].between(SLICE_START_YEAR, SLICE_END_YEAR)
].reset_index(drop=True)
print(f"Yearly candidate PMIDs in slice [{SLICE_START_YEAR}, {SLICE_END_YEAR}]: {len(candidate_df):,}")
resume_df = None
if OUTPUT_UNIFIED_DECISIONS_PATH.exists():
    resume_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)
if RUN_LLM_FILTER:
    llm = OpenAILanguageModel(OpenAIConfig(model="gpt-4o-mini", max_tokens=256, temperature=0))
    unified_df = run_unified_llm_filter(
        candidate_df,
        llm,
        email=PUBMED_EMAIL,
        system_prompt=UNIFIED_SYSTEM_PROMPT,
        output_path=str(OUTPUT_UNIFIED_DECISIONS_PATH),
        resume_from=resume_df,
        auto_resume_from_output=True,
        llm_max_workers=LLM_MAX_WORKERS,
        llm_batch_size=LLM_BATCH_SIZE,
        abstract_fetch_batch_size=ABSTRACT_FETCH_BATCH_SIZE,
        prefetch_abstracts=PREFETCH_ABSTRACTS,
        failed_retry_path=str(FAILED_RETRY_PATH),
        max_failure_retries=MAX_FAILURE_RETRIES,
        abstract_cache_path=str(ABSTRACT_CACHE_PATH),
        max_llm_calls=MAX_LLM_CALLS,
        progress_every=UNIFIED_PROGRESS_EVERY,
        verbose=1,
    )
else:
    unified_df = resume_df if resume_df is not None else pd.DataFrame()
    if unified_df.empty:
        raise FileNotFoundError(
            f"No unified decisions file found at {OUTPUT_UNIFIED_DECISIONS_PATH}. "
            "Run with RUN_LLM_FILTER=True at least once."
        )
print(unified_df["unified_decision"].value_counts(dropna=False))
ABSTRACT_FETCH_BATCH_SIZE = 200
PREFETCH_ABSTRACTS = True


## Export Canonical Screened Yearly Dataset
Includes decisions `yes` + `uncertain` by default.


In [ ]:
yearly_screened_df = build_yearly_screened_pairs(
    yearly_amr_df,
    unified_df,
    include_decisions=("yes", "uncertain"),
    output_path=str(OUTPUT_YEARLY_SCREEN_PATH),
    merge_with_existing_output=True,
)

print(f"Screened yearly rows: {len(yearly_screened_df):,}")
print(f"Unique pairs: {yearly_screened_df[['pathogen','antibiotic']].drop_duplicates().shape[0]:,}")
print(f"Year span: {int(yearly_screened_df['year'].min())} - {int(yearly_screened_df['year'].max())}")
yearly_screened_df.head(10)

